In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(f"Loaded: {df.shape}")
print(f"Default rate before processing: {df['SeriousDlqin2yrs'].mean()*100:.2f}%")

Loaded: (150000, 11)
Default rate before processing: 6.68%


In [3]:
# Some rows have extreme values like age=0 or age=109
# These are data entry errors — we fix them

print("OUTLIER CHECK")
print("=" * 40)
print(f"Age = 0: {(df['age'] == 0).sum()} rows")
print(f"Age > 100: {(df['age'] > 100).sum()} rows")
print(f"RevolvingUtilization > 1: {(df['RevolvingUtilizationOfUnsecuredLines'] > 1).sum()} rows")
print(f"DebtRatio > 1: {(df['DebtRatio'] > 1).sum()} rows")

# Remove rows where age is 0 — clearly wrong
df = df[df['age'] > 0]

# Cap RevolvingUtilization at 1 — above 1 means over credit limit
df['RevolvingUtilizationOfUnsecuredLines'] = df[
    'RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.0)

# Cap extreme late payment values at 10 — anything above is data error
late_cols = ['NumberOfTime30-59DaysPastDueNotWorse',
             'NumberOfTime60-89DaysPastDueNotWorse',
             'NumberOfTimes90DaysLate']
for col in late_cols:
    df[col] = df[col].clip(upper=10)

print(f"\nAfter cleaning: {df.shape}")

OUTLIER CHECK
Age = 0: 1 rows
Age > 100: 13 rows
RevolvingUtilization > 1: 3321 rows
DebtRatio > 1: 35137 rows

After cleaning: (149999, 11)


In [4]:
# MonthlyIncome: 19.82% missing — use median by age group
# This is smarter than global median because income varies heavily by age

df['age_group'] = pd.cut(df['age'], 
                          bins=[0, 25, 35, 45, 55, 65, 120],
                          labels=['<25', '25-35', '35-45', '45-55', '55-65', '65+'])

print("Median Income by Age Group:")
print(df.groupby('age_group')['MonthlyIncome'].median().round(0))

# Fill missing MonthlyIncome with age group median
df['MonthlyIncome'] = df.groupby('age_group')['MonthlyIncome'].transform(
    lambda x: x.fillna(x.median())
)

# NumberOfDependents: 2.62% missing — simple median fill
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(
    df['NumberOfDependents'].median()
)

# Drop age_group helper column
df = df.drop('age_group', axis=1)

# Verify no missing values remain
print(f"\nMissing values remaining: {df.isnull().sum().sum()}")
print("✅ All missing values handled" if df.isnull().sum().sum() == 0 
      else "⚠️ Still have missing values")

Median Income by Age Group:
age_group
<25      1600.0
25-35    3876.0
35-45    5611.0
45-55    6267.0
55-65    6176.0
65+      4968.0
Name: MonthlyIncome, dtype: float64

Missing values remaining: 0
✅ All missing values handled


In [5]:
# Create new features that capture risk signals better
# This is what separates a good DS from a basic one

# Total times ever late (combines all late payment columns)
df['TotalTimesLate'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] + 
                         df['NumberOfTime60-89DaysPastDueNotWorse'] + 
                         df['NumberOfTimes90DaysLate'])

# Income to debt burden ratio
df['IncomeDebtBurden'] = df['MonthlyIncome'] / (df['DebtRatio'] + 1)

# Credit line utilization per open account
df['UtilizationPerAccount'] = (df['RevolvingUtilizationOfUnsecuredLines'] / 
                                (df['NumberOfOpenCreditLinesAndLoans'] + 1))

# Flag for serious delinquency (ever 90+ days late)
df['EverSeriouslyLate'] = (df['NumberOfTimes90DaysLate'] > 0).astype(int)

print("New features created:")
print(f"TotalTimesLate — range: {df['TotalTimesLate'].min()} to {df['TotalTimesLate'].max()}")
print(f"IncomeDebtBurden — median: {df['IncomeDebtBurden'].median():.0f}")
print(f"UtilizationPerAccount — median: {df['UtilizationPerAccount'].median():.3f}")
print(f"EverSeriouslyLate — {df['EverSeriouslyLate'].sum():,} applicants flagged")
print(f"\nFinal feature count: {df.shape[1]} columns")

New features created:
TotalTimesLate — range: 0 to 30
IncomeDebtBurden — median: 3376
UtilizationPerAccount — median: 0.016
EverSeriouslyLate — 8,338 applicants flagged

Final feature count: 15 columns


In [6]:
# Separate features and target
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

print(f"Features: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts()}")

# Split — stratify ensures same default ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # ← important for imbalanced data
)

print(f"\nTrain size: {X_train.shape[0]:,}")
print(f"Test size:  {X_test.shape[0]:,}")
print(f"Train default rate: {y_train.mean()*100:.2f}%")
print(f"Test default rate:  {y_test.mean()*100:.2f}%")

Features: 14
Target distribution:
SeriousDlqin2yrs
0    139973
1     10026
Name: count, dtype: int64

Train size: 119,999
Test size:  30,000
Train default rate: 6.68%
Test default rate:  6.68%


In [7]:
# SMOTE = Synthetic Minority Oversampling Technique
# Creates synthetic examples of the minority class (defaulters)
# Only applied to TRAINING data — never to test data

print(f"Before SMOTE - Train shape: {X_train.shape}")
print(f"Before SMOTE - Default rate: {y_train.mean()*100:.2f}%")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE - Train shape: {X_train_balanced.shape}")
print(f"After SMOTE - Default rate: {y_train_balanced.mean()*100:.2f}%")
print(f"New class distribution:\n{pd.Series(y_train_balanced).value_counts()}")

Before SMOTE - Train shape: (119999, 14)
Before SMOTE - Default rate: 6.68%

After SMOTE - Train shape: (223956, 14)
After SMOTE - Default rate: 50.00%
New class distribution:
SeriousDlqin2yrs
0    111978
1    111978
Name: count, dtype: int64


In [8]:
# StandardScaler: transforms features to mean=0, std=1
# Important for Logistic Regression — less critical for tree models
# but good practice to do consistently

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)  # ← use transform only, not fit_transform

print("Scaling complete ✅")
print(f"Train scaled shape: {X_train_scaled.shape}")
print(f"Test scaled shape:  {X_test_scaled.shape}")

# Quick sanity check
print(f"\nMean of first feature (should be ~0): {X_train_scaled[:,0].mean():.4f}")
print(f"Std of first feature (should be ~1):  {X_train_scaled[:,0].std():.4f}")

Scaling complete ✅
Train scaled shape: (223956, 14)
Test scaled shape:  (30000, 14)

Mean of first feature (should be ~0): -0.0000
Std of first feature (should be ~1):  1.0000


In [9]:
# Save all preprocessed data and the scaler
# Notebook 3 loads these directly — no reprocessing needed

os.makedirs('../models', exist_ok=True)

# Save arrays
np.save('../models/X_train_scaled.npy', X_train_scaled)
np.save('../models/X_test_scaled.npy', X_test_scaled)
np.save('../models/y_train_balanced.npy', y_train_balanced)
np.save('../models/y_test.npy', y_test.values)

# Save scaler — needed by the app to scale user input
joblib.dump(scaler, '../models/scaler.pkl')

# Save feature names — needed for SHAP plots
feature_names = X.columns.tolist()
joblib.dump(feature_names, '../models/feature_names.pkl')

print("Saved to models/ folder:")
print("  ✅ X_train_scaled.npy")
print("  ✅ X_test_scaled.npy")
print("  ✅ y_train_balanced.npy")
print("  ✅ y_test.npy")
print("  ✅ scaler.pkl")
print("  ✅ feature_names.pkl")
print(f"\nFeatures going into model: {feature_names}")

Saved to models/ folder:
  ✅ X_train_scaled.npy
  ✅ X_test_scaled.npy
  ✅ y_train_balanced.npy
  ✅ y_test.npy
  ✅ scaler.pkl
  ✅ feature_names.pkl

Features going into model: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'TotalTimesLate', 'IncomeDebtBurden', 'UtilizationPerAccount', 'EverSeriouslyLate']
